<a href="https://colab.research.google.com/github/annanya-sharma1002/AI_for_Health/blob/main/AI_for_health.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def load_signal(file):
    df = pd.read_csv(
        file,
        sep=";",
        skiprows=7,
        names=["timestamp", "value"]
    )

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        format="%d.%m.%Y %H:%M:%S,%f"
    )

    return df
nasal = load_signal("/content/Flow - 30-05-2024.txt")
thoracic = load_signal("/content/Thorac - 30-05-2024.txt")
spo2 = load_signal("/content/SPO2 - 30-05-2024.txt")

#load event file
events = pd.read_csv(
    "/content/Flow Events - 30-05-2024.txt",
    sep=";",
    skiprows=5,
    names=["time_range", "duration", "event", "stage"]
)
#split the start and end time
events[["start", "end"]] = events["time_range"].str.split("-", expand=True)
#add date to the end timestamp
events["date"] = events["start"].str[:10]
events["end"] = events["date"] + " " + events["end"]
#convert to datetime
events["start"] = pd.to_datetime(
    events["start"],
    format="%d.%m.%Y %H:%M:%S,%f"
)
events["end"] = pd.to_datetime(
    events["end"],
    format="%d.%m.%Y %H:%M:%S,%f"
)

#define time window
window = pd.Timedelta(minutes=5)

start_time = nasal["timestamp"].iloc[0]
end_time = nasal["timestamp"].iloc[-1]

colors = {
    "Hypopnea": "orange",
    "Obstructive Apnea": "red"
}


 #generate pdf
with PdfPages("sleep_report.pdf") as pdf:

    current_start = start_time

    while current_start < end_time:

        current_end = current_start + window

        nasal_seg = nasal[(nasal["timestamp"] >= current_start) &
                          (nasal["timestamp"] < current_end)]

        thoracic_seg = thoracic[(thoracic["timestamp"] >= current_start) &
                                (thoracic["timestamp"] < current_end)]

        spo2_seg = spo2[(spo2["timestamp"] >= current_start) &
                        (spo2["timestamp"] < current_end)]

        fig, axs = plt.subplots(3, 1, figsize=(20,8), sharex=True)

        axs[0].plot(nasal_seg["timestamp"], nasal_seg["value"], color="blue")
        axs[0].set_title("Nasal Flow")
        axs[0].set_ylabel("Flow")

        axs[1].plot(thoracic_seg["timestamp"], thoracic_seg["value"], color="orange")
        axs[1].set_title("Thoracic/Abdominal Respiration")
        axs[1].set_ylabel("Movement")

        axs[2].plot(spo2_seg["timestamp"], spo2_seg["value"], color="green")
        axs[2].set_title("SpO₂")
        axs[2].set_ylabel("%")

        for _, e in events.iterrows():
            if e["start"] < current_end and e["end"] > current_start:

              color = colors.get(e["event"], "gray")

              for ax in axs:
                ax.axvspan(
                    e["start"],
                    e["end"],
                    color=color,
                    alpha=0.3
                )

        fig.suptitle(f"{current_start} to {current_end}")

        plt.tight_layout()

        pdf.savefig(fig)
        plt.close()

        current_start = current_end
